# Topic 24 — TF-IDF
### ⭐ This should be one of your strongest NLP concepts. Theory → formula → from-scratch → sklearn.

Plain word counts (Bag of Words, Topic 22) have a flaw: very common words ("the", "you", "is") get
huge counts everywhere and drown out rarer, more distinctive words. **TF-IDF** (Term
Frequency–Inverse Document Frequency) fixes this by *down-weighting* words that appear in almost
every document, and *up-weighting* words that are frequent in one document but rare overall —
exactly the words that actually distinguish that document from the rest.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. The formula, piece by piece

```text
TF(word, doc)  = (count of word in doc) / (total words in doc)
IDF(word)      = log( total_documents / (1 + documents_containing_word) )
TF-IDF(word, doc) = TF(word, doc) * IDF(word)
```

- **TF** rewards words that appear often *within this document*.
- **IDF** punishes words that appear in *many documents* (they're not distinctive) and rewards rare ones.
  The `+1` in the denominator avoids division by zero for words appearing in zero documents.
- Multiplying them together: a word scores high only if it's frequent HERE but rare EVERYWHERE ELSE.

In [ ]:
docs = [
    "you are stupid and worthless",
    "i hate you so much",
    "great job today team",
    "you did a great job",
]

def compute_tf(doc, vocab):
    words = doc.lower().split()
    return np.array([words.count(w) / len(words) for w in vocab])

def compute_idf(docs, vocab):
    n_docs = len(docs)
    idf = []
    for w in vocab:
        doc_count = sum(1 for doc in docs if w in doc.lower().split())
        idf.append(np.log(n_docs / (1 + doc_count)))
    return np.array(idf)

vocab = sorted(set(word for doc in docs for word in doc.lower().split()))
print("vocabulary:", vocab)

idf = compute_idf(docs, vocab)
print("\nIDF per word:")
for w, v in zip(vocab, idf):
    print(f"  {w:<10} idf={v:.3f}")
# Notice "you" (appears in docs 0, 1, and 3) has a LOW idf -- it's common, not distinctive.
# A rare word appearing in only one document gets a HIGH idf.

In [ ]:
tfidf_matrix = np.array([compute_tf(doc, vocab) * idf for doc in docs])
tfidf_df = pd.DataFrame(tfidf_matrix, columns=vocab, index=[f"doc{i}" for i in range(len(docs))])
print(tfidf_df.round(3))
# For each row (document), the highest-scoring words are the ones that BEST distinguish that
# document from the others -- look at doc0 ("stupid", "worthless") vs doc2/doc3 ("great", "job").

## 2. sklearn's `TfidfVectorizer`

Does the same computation (with a couple of implementation refinements — smoothing and L2
normalization by default), directly from raw text, and it's what you'll actually use in practice.

In [ ]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(docs)

sklearn_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out(),
                           index=[f"doc{i}" for i in range(len(docs))])
print(sklearn_df.round(3))
# Values won't match your from-scratch version exactly (sklearn adds smoothing + L2-normalizes
# each row so all document vectors have length 1), but the RELATIVE pattern should look similar:
# common words score low, distinctive words score high.

## 3. L2 normalization — why sklearn rows have unit length

sklearn normalizes each document's TF-IDF vector to have length 1 (L2 norm = 1). This makes
documents of different LENGTHS comparable — a long document and a short document about the same
topic end up with similar-shaped vectors, rather than the long one just having bigger numbers everywhere.

In [ ]:
row_norms = np.linalg.norm(X_tfidf.toarray(), axis=1)
print("L2 norm of each document's TF-IDF vector:", row_norms)
# Should all be very close to 1.0

## 4. Comparing raw counts vs TF vs TF-IDF, side by side

In [ ]:
count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(docs).toarray()

comparison = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "raw_count_doc0": X_counts[0],
    "tfidf_doc0": X_tfidf.toarray()[0],
}).sort_values("tfidf_doc0", ascending=False)
print(comparison)
# "you" might have a nonzero raw count but a relatively LOW tfidf score (it's common across docs).
# "stupid"/"worthless" should have HIGH tfidf scores -- they're what makes doc0 distinctive.

## 5. TF-IDF with n-grams (combining Topics 23 & 24)

In [ ]:
vec_ngram_tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
X_ngram_tfidf = vec_ngram_tfidf.fit_transform(docs)
print("vocabulary (unigrams + bigrams):", vec_ngram_tfidf.get_feature_names_out())
print("shape:", X_ngram_tfidf.shape)

## 6. Finding the most important words for a specific document

In [ ]:
def top_tfidf_words(vectorizer, X_tfidf, doc_index, top_n=5):
    row = X_tfidf.toarray()[doc_index]
    feature_names = vectorizer.get_feature_names_out()
    top_idx = np.argsort(row)[::-1][:top_n]
    return [(feature_names[i], round(row[i], 3)) for i in top_idx if row[i] > 0]

for i, doc in enumerate(docs):
    print(f"doc{i} ('{doc}'):")
    print("  top words:", top_tfidf_words(vectorizer, X_tfidf, i))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 2 more documents to `docs` that reuse some existing words but introduce 2-3 brand-new ones,
#    recompute IDF from scratch, and check that the brand-new words get high IDF.
# 2. Use TfidfVectorizer(max_features=5) on bigger_docs from Topic 23 and inspect which 5 words
#    it kept (hint: highest average TF-IDF across the corpus, roughly).
# 3. Compute cosine similarity (1 - scipy.spatial.distance.cosine, or use sklearn.metrics.pairwise
#    .cosine_similarity) between two TF-IDF document vectors -- this is how you'd measure how
#    SIMILAR two comments are, a common step before/instead of clustering (Topic 18).
# 4. In your own words: why does multiplying TF by IDF (rather than using either alone) produce
#    better features for classification than raw word counts?

---
### Next up: **Topic 25 — Classical NLP Classification** (TF-IDF + LogReg/NB/SVM/RF — directly relevant to your paper).

Say "next" when you're ready.